# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo1"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo1" /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo1/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Oct 05 19:13:29 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671199,35.9,1479655,79.1,1479655,79.1
Vcells,1242615,9.5,8388608,64.0,1978693,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia

## 4.02 Árboles Azarosos + Grid Search

Esta versión conserva la lógica del `z420_ArbolesAzarosos` original:

- se usa todo `202107` como entrenamiento;
- se usa `202109` como período futuro;
- se fija una sola vez la `semilla_primigenia` para cada combinación;
- cada ensemble contiene 32 árboles;
- cada árbol utiliza un subconjunto aleatorio de campos;
- se acumulan las probabilidades de `BAJA+2`;
- el corte final es `(1/40) * cantidad_de_arboles`;
- cada combinación genera un CSV y se sube inmediatamente a Kaggle.

No hay partición 70/30 ni ranking local: Kaggle evalúa directamente todas las configuraciones.


In [4]:
PARAM <- list()

# SU semilla primigenia
PARAM$semilla_primigenia <- 149911

# cantidad de árboles por ensemble
PARAM$num_trees_max <- 32L

# grilla de hiperparámetros
PARAM$grid_feature_fraction <- c(0.3, 0.5, 0.7)
PARAM$grid_maxdepth <- c(6, 7, 8, 9)
PARAM$grid_cp <- -1
PARAM$grid_minsplit <- c(400, 600, 800, 1000)
PARAM$grid_ratio_bucket <- c(1/3)

# competencia Kaggle
PARAM$competencia <- "labo-1-virtual-inicial"

# si la corrida se corta, cambiar este valor
# por la siguiente iteración que falta
PARAM$iter_inicio <- 1L


In [5]:
# cantidad total de combinaciones
qcombinaciones <- (
  length(PARAM$grid_feature_fraction) *
  length(PARAM$grid_maxdepth) *
  length(PARAM$grid_cp) *
  length(PARAM$grid_minsplit) *
  length(PARAM$grid_ratio_bucket)
)

cat(
  "Combinaciones:",
  qcombinaciones,
  "\n"
)

cat(
  "Árboles totales:",
  qcombinaciones * PARAM$num_trees_max,
  "\n"
)


Combinaciones: 48 
Árboles totales: 1536 


In [6]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")

experimento <- "exp4025_gridsearch_kaggle_48"
dir.create(
  experimento,
  showWarnings = FALSE
)

setwd(
  paste0(
    "/content/buckets/b1/exp/",
    experimento
  )
)

getwd()


[1] "/home/ds/buckets/b1/exp/exp4025_gridsearch_kaggle_48"

In [7]:
# lectura del dataset
dataset <- fread(
  "/content/datasets/dataset_pequeno.csv"
)

# igual que en el notebook original
dtrain <- dataset[
  foto_mes == 202107
]

dfuture <- dataset[
  foto_mes == 202109
]

# arreglo clase_ternaria
dfuture[
  ,
  clase_ternaria := NA
]

# mismos campos candidatos del z420 original
campos_buenos <- copy(
  setdiff(
    colnames(dtrain),
    c("clase_ternaria")
  )
)

cat(
  "202107:",
  nrow(dtrain),
  "registros\n"
)

cat(
  "202109:",
  nrow(dfuture),
  "registros\n"
)

cat(
  "Campos candidatos:",
  length(campos_buenos),
  "\n"
)


202107: 164479 registros
202109: 165076 registros
Campos candidatos: 154 


### Generar el ensemble de 32 árboles

Replica la lógica del `z420` original.

La semilla se fija una sola vez antes del loop. Luego cada `sample()` consume el siguiente estado de la secuencia pseudoaleatoria, generando 32 subconjuntos distintos pero reproducibles.


In [8]:
GenerarEnsemble32 <- function(
  param_rpart,
  feature_fraction
) {

  tb_prediccion <- dfuture[
    ,
    list(
      numero_de_cliente
    )
  ]

  tb_prediccion[
    ,
    prob_acumulada := 0
  ]

  # una sola semilla antes de los 32 árboles
  set.seed(
    PARAM$semilla_primigenia
  )

  qty_campos_a_utilizar <- as.integer(
    length(campos_buenos) *
      feature_fraction
  )

  for (
    arbolito in seq(
      PARAM$num_trees_max
    )
  ) {

    # elijo los campos al azar
    campos_random <- sample(
      campos_buenos,
      qty_campos_a_utilizar
    )

    campos_random <- paste(
      campos_random,
      collapse = " + "
    )

    formulita <- paste0(
      "clase_ternaria ~ ",
      campos_random
    )

    # genero el árbol
    modelo <- rpart(
      formulita,
      data = dtrain,
      xval = 0,
      control = param_rpart
    )

    # predigo 202109
    prediccion <- predict(
      modelo,
      dfuture,
      type = "prob"
    )

    # acumulo probabilidad
    tb_prediccion[
      ,
      prob_acumulada :=
        prob_acumulada +
          prediccion[, "BAJA+2"]
    ]

    if (
      arbolito %% 8 == 0 ||
      arbolito == PARAM$num_trees_max
    ) {

      cat(
        "   árboles:",
        arbolito,
        "de",
        PARAM$num_trees_max,
        "\n"
      )

      flush.console()
    }
  }

  # mismo criterio del notebook original
  umbral_corte <- (
    1 / 40
  ) * PARAM$num_trees_max

  tb_prediccion[
    ,
    Predicted :=
      as.numeric(
        prob_acumulada >
          umbral_corte
      )
  ]

  return(
    tb_prediccion[
      ,
      list(
        numero_de_cliente,
        Predicted
      )
    ]
  )
}


## 4.03 Grid Search + subida inmediata a Kaggle

Grilla utilizada:

- `feature_fraction`: 0.3, 0.5, 0.7
- `maxdepth`: 6, 7, 8, 9
- `cp`: -1
- `minsplit`: 400, 600, 800, 1000
- `minbucket`: `round(minsplit * 1/3)`

Total:

`3 × 4 × 1 × 4 × 1 = 48 combinaciones`

Cada combinación genera 32 árboles:

`48 × 32 = 1536 árboles`


In [ ]:
tb_subidas <- data.table()

iter <- 0L

for (
  vmax_depth in PARAM$grid_maxdepth
) {

  for (
    vcp in PARAM$grid_cp
  ) {

    for (
      vmin_split in PARAM$grid_minsplit
    ) {

      for (
        vratio_bucket in PARAM$grid_ratio_bucket
      ) {

        for (
          vfeature_fraction in
            PARAM$grid_feature_fraction
        ) {

          iter <- iter + 1L

          # reanudación
          if (
            iter < PARAM$iter_inicio
          ) {
            next
          }

          vmin_bucket <- as.integer(
            round(
              vmin_split *
                vratio_bucket
            )
          )

          cat(
            "\n========================================\n",
            "Iteración",
            iter,
            "de",
            qcombinaciones,
            "\n",
            "cp =", vcp,
            "| maxdepth =", vmax_depth,
            "| minsplit =", vmin_split,
            "| minbucket =", vmin_bucket,
            "| feature_fraction =",
            vfeature_fraction,
            "\n",
            "========================================\n"
          )

          flush.console()

          param_rpart <- list(
            cp = vcp,
            minsplit = vmin_split,
            minbucket = vmin_bucket,
            maxdepth = vmax_depth
          )

          # entrena 32 árboles y genera predicción
          submission <- GenerarEnsemble32(
            param_rpart =
              param_rpart,
            feature_fraction =
              vfeature_fraction
          )

          archivo_kaggle <- paste0(
            "KA_GRID_",
            sprintf(
              "%03d",
              iter
            ),
            ".csv"
          )

          fwrite(
            submission,
            file = archivo_kaggle,
            sep = ","
          )

          # mensaje identificador
          mensaje <- paste0(
            "-m 'iter=", iter,
            " cp=", vcp,
            " minsplit=", vmin_split,
            " minbucket=", vmin_bucket,
            " maxdepth=", vmax_depth,
            " feature_fraction=",
            vfeature_fraction,
            "'"
          )

          comando <-
            "kaggle competitions submit"

          competencia <- paste(
            "-c",
            PARAM$competencia
          )

          arch <- paste(
            "-f",
            archivo_kaggle
          )

          linea <- paste(
            comando,
            competencia,
            arch,
            mensaje
          )

          cat(
            "Subiendo:",
            archivo_kaggle,
            "\n"
          )

          salida <- tryCatch(
            {
              system(
                linea,
                intern = TRUE
              )
            },
            error = function(e) {
              paste(
                "ERROR:",
                e$message
              )
            }
          )

          cat(
            paste(
              salida,
              collapse = "\n"
            ),
            "\n"
          )

          tb_subidas <- rbind(
            tb_subidas,
            data.table(
              iter = iter,
              cp = vcp,
              maxdepth = vmax_depth,
              minsplit = vmin_split,
              minbucket = vmin_bucket,
              ratio_bucket = vratio_bucket,
              feature_fraction =
                vfeature_fraction,
              archivo =
                archivo_kaggle,
              respuesta_kaggle =
                paste(
                  salida,
                  collapse = " "
                )
            ),
            fill = TRUE
          )

          # persistencia después de cada combinación
          fwrite(
            tb_subidas,
            file =
              "gridsearch_subidas_kaggle.csv"
          )

          Sys.sleep(3)

          gc(
            verbose = FALSE
          )
        }
      }
    }
  }
}



 Iteración 1 de 48 
 cp = -1 | maxdepth = 6 | minsplit = 400 | minbucket = 133 | feature_fraction = 0.3 
   árboles: 8 de 32 
   árboles: 16 de 32 
   árboles: 24 de 32 
   árboles: 32 de 32 
Subiendo: KA_GRID_001.csv 
96 submissions remaining today.
Successfully submitted to Labo 1 Virtual Inicial 

 Iteración 2 de 48 
 cp = -1 | maxdepth = 6 | minsplit = 400 | minbucket = 133 | feature_fraction = 0.5 
   árboles: 8 de 32 
   árboles: 16 de 32 
   árboles: 24 de 32 
   árboles: 32 de 32 
Subiendo: KA_GRID_002.csv 
95 submissions remaining today.
Successfully submitted to Labo 1 Virtual Inicial 

 Iteración 3 de 48 
 cp = -1 | maxdepth = 6 | minsplit = 400 | minbucket = 133 | feature_fraction = 0.7 
   árboles: 8 de 32 
   árboles: 16 de 32 
   árboles: 24 de 32 
   árboles: 32 de 32 
Subiendo: KA_GRID_003.csv 
94 submissions remaining today.
Successfully submitted to Labo 1 Virtual Inicial 

 Iteración 4 de 48 
 cp = -1 | maxdepth = 6 | minsplit = 600 | minbucket = 200 | feature_frac

## 4.04 Recuperar scores y construir ranking Kaggle

Después de completar las submissions se consulta Kaggle y se cruza el `publicScore` con los hiperparámetros de cada archivo.


In [ ]:
salida_kaggle <- system(
  paste(
    "kaggle competitions submissions -c",
    PARAM$competencia,
    "--csv"
  ),
  intern = TRUE
)

historial_kaggle <- fread(
  text = paste(
    salida_kaggle,
    collapse = "\n"
  )
)

historial_kaggle


In [ ]:
scores_kaggle <- historial_kaggle[
  ,
  .(
    archivo = fileName,
    estado = status,
    score_kaggle = publicScore
  )
]

ranking_kaggle <- merge(
  tb_subidas,
  scores_kaggle,
  by = "archivo",
  all.x = TRUE
)

ranking_kaggle[
  ,
  score_kaggle :=
    as.numeric(
      score_kaggle
    )
]

ranking_kaggle <- ranking_kaggle[
  order(
    -score_kaggle
  )
]

ranking_kaggle[
  1:min(20, .N)
]


In [ ]:
fwrite(
  ranking_kaggle,
  file =
    "gridsearch_ranking_kaggle.csv"
)


In [ ]:
format(
  Sys.time(),
  "%a %b %d %X %Y"
)
